# f6_m00b_preparacion_app.ipynb

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 6 — Interpretabilidad y Evaluación Final |
| **Módulo** | M00b — Preparación para la app |

---

## 🎯 Qué hace

Construye `meta_test_app.parquet` — un único fichero enriquecido con todo lo que
la app Streamlit (Fase 7) necesita del conjunto de test, evitando joins en tiempo
de ejecución. Fusiona `meta_test.parquet` (metadatos) con las features originales
de `X_test.parquet` y los flags `_missing` de `X_test_prep.parquet`. Debe ejecutarse
**después** de `f6_m00_preparacion` (que genera `meta_test.parquet`).

## 📋 Requisitos

- `data/06_evaluacion/meta_test.parquet` — metadatos del test (generado en f6_m00_preparacion)
- `data/05_modelado/X_test_prep.parquet` — features preparadas, aporta los flags `_missing`
- `data/05_modelado/X_test.parquet` — features originales del test
- Entorno: `tfm_abandono` (pandas)

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `data/06_evaluacion/meta_test_app.parquet` | Fichero enriquecido para la app (Fase 7): metadatos + features originales + flags `_missing`. 6.725 filas × 35 columnas |

## 🔄 Flujo

```
data/06_evaluacion/meta_test.parquet      ┐
data/05_modelado/X_test.parquet           ├─→ concat por índice → meta_test_app
data/05_modelado/X_test_prep.parquet      ┘
    ↓ verificación de los 3 ficheros necesarios e índices idénticos
    ↓ construcción de df_app (metadatos + features + flags _missing)
    ↓ verificación de casos canónicos
    → data/06_evaluacion/meta_test_app.parquet
```

## ➡️ Siguiente

`f6_m00c_export_probs.ipynb` — export de probabilidades del modelo sobre el test

In [1]:
# ============================================================================
# CELDA 1: IMPORTS Y RUTAS
# ============================================================================
import sys
from pathlib import Path

# Detección robusta de ROOT subiendo niveles hasta encontrar src/
ROOT = Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd

print(f"ROOT: {ROOT}")
print(f"Python: {sys.version.split()[0]}")

ROOT: c:\PRUEBAS\AU_UJI_v2_RUTA_B
Python: 3.11.14


In [2]:
# ============================================================================
# CELDA 2: CARGA DE FICHEROS FUENTE
# ============================================================================
RUTA_META   = ROOT / "data" / "06_evaluacion" / "meta_test.parquet"
RUTA_X_PREP = ROOT / "data" / "05_modelado"   / "X_test_prep.parquet"
RUTA_X_TEST = ROOT / "data" / "05_modelado"   / "X_test.parquet"

for ruta in [RUTA_META, RUTA_X_PREP, RUTA_X_TEST]:
    assert ruta.exists(), f"❌ No encontrado: {ruta}"

meta   = pd.read_parquet(RUTA_META)
X_prep = pd.read_parquet(RUTA_X_PREP)
X_test = pd.read_parquet(RUTA_X_TEST)

# Verificar índices idénticos
assert list(meta.index) == list(X_prep.index) == list(X_test.index), \
    "❌ Los índices no coinciden — re-ejecuta f6_m00_preparacion.ipynb"

print(f"meta_test:   {meta.shape}")
print(f"X_test_prep: {X_prep.shape}")
print(f"X_test:      {X_test.shape}")
print("✅ Ficheros cargados e índices verificados")

meta_test:   (6725, 14)
X_test_prep: (6725, 27)
X_test:      (6725, 27)
✅ Ficheros cargados e índices verificados


In [3]:
# ============================================================================
# CELDA 3: CONSTRUIR meta_test_app
# ============================================================================
# Columnas ya en meta (no duplicar)
cols_meta = set(meta.columns)

# Features de X_test originales — excluir las que ya están en meta
# y excluir _missing (se añaden desde X_prep para consistencia)
cols_nuevas  = [c for c in X_test.columns
                if c not in cols_meta and not c.endswith("_missing")]

# Flags _missing desde X_prep
missing_cols = [c for c in X_prep.columns if c.endswith("_missing")]

# Fusión por índice (segura porque índices son idénticos)
df_app = pd.concat([meta, X_test[cols_nuevas], X_prep[missing_cols]], axis=1)

# Verificar sin duplicados
assert df_app.columns.duplicated().sum() == 0, "❌ Columnas duplicadas"

print(f"Shape final: {df_app.shape}")
print(f"Columnas ({len(df_app.columns)}):")
print(df_app.columns.tolist())
print()
nulos = df_app.isnull().sum()
nulos_reales = nulos[nulos > 0]
if len(nulos_reales):
    print("Nulos por columna:")
    print(nulos_reales)
else:
    print("✅ Sin nulos inesperados")

Shape final: (6725, 35)
Columnas (35):
['titulacion', 'rama', 'sexo', 'pais_nombre', 'provincia', 'via_acceso', 'abandono', 'per_id_ficticio', 'curso_aca_ini', 'curso_aca', 'vive_fuera', 'cupo', 'n_titulaciones', 'flag_cautela', 'cred_superados_anio_1er', 'nota_1er_anio', 'nota_acceso', 'nota_selectividad', 'tasa_abandono_titulacion', 'universidad_origen', 'n_anios_beca', 'anios_sin_beca', 'situacion_laboral', 'n_anios_trabajando', 'max_pagos', 'indicador_interrupcion', 'orden_preferencia', 'cred_repetidos', 'tasa_repeticion', 'edad_entrada', 'anios_gap', 'n_anios_sin_notas', 'nota_1er_anio_missing', 'nota_acceso_missing', 'nota_selectividad_missing']

Nulos por columna:
cupo    709
dtype: int64


In [4]:
# ============================================================================
# CELDA 4: VERIFICACIÓN DE CASOS CANÓNICOS
# ============================================================================
# 5 perfiles seleccionados para la sección de ejemplos reales de la app.
# Si el conjunto de test cambia, algún índice podría dejar de existir: en ese
# caso se avisa con un mensaje explícito y se continúa, sin detener el notebook.
# ============================================================================
CASOS_CANONICOS = {
    "C1 — Ing. Informática (FP, abandona)":          15872,
    "C2 — Medicina (nota 11.94, no abandona)":       11906,
    "C3 — Comunicación (nota alta, abandona)":        14957,
    "C4 — Ing. Informática (mujer, no abandona)":     32472,
    "C5 — Derecho (mayor 25, trabaja, abandona)":      7176,
}

cols_check = ["titulacion", "rama", "sexo", "via_acceso",
              "nota_acceso", "nota_1er_anio", "edad_entrada",
              "n_anios_trabajando", "abandono"]

n_encontrados = 0
n_omitidos = 0

for nombre, idx in CASOS_CANONICOS.items():
    # Red de seguridad: si el test ha cambiado, el índice podría no existir.
    if idx not in df_app.index:
        n_omitidos += 1
        print(f"\n⚠️  CASO CANÓNICO NO ENCONTRADO")
        print(f"    Caso:   {nombre}")
        print(f"    Índice: {idx} — no existe en el conjunto de test actual")
        print(f"    El test actual tiene {len(df_app)} filas "
              f"(rango de índices {df_app.index.min()} – {df_app.index.max()}).")
        print(f"    Causa probable: el conjunto de test ha cambiado desde que se")
        print(f"                    fijaron estos índices (p.ej. un refiltrado de")
        print(f"                    datos en Fase 1-5 o una nueva ejecución del split).")
        print(f"    Acción: actualizar el diccionario CASOS_CANONICOS de esta celda")
        print(f"            con índices que sí existan en df_app.index.")
        print(f"    → El caso se omite; los demás casos y el guardado del parquet continúan.")
        continue

    n_encontrados += 1
    row = df_app.loc[idx]
    icono = "🔴 ABANDONA" if row["abandono"] == 1 else "🟢 NO ABANDONA"
    print(f"\n{icono} | {nombre}")
    for c in cols_check:
        if c in row.index:
            val = round(row[c], 3) if isinstance(row[c], float) else row[c]
            print(f"  {c}: {val}")

# Resumen final dinámico — de un vistazo se ve el estado de todos los casos
print(f"\n{'=' * 60}")
n_total = len(CASOS_CANONICOS)
if n_omitidos == 0:
    print(f"RESUMEN CASOS CANÓNICOS: {n_total} definidos · {n_encontrados} encontrados ✅")
else:
    print(f"RESUMEN CASOS CANÓNICOS: {n_total} definidos · "
          f"{n_encontrados} encontrados ✅ · {n_omitidos} omitidos ⚠️")
    print(f"  Revisar el diccionario CASOS_CANONICOS (ver avisos arriba).")



🔴 ABANDONA | C1 — Ing. Informática (FP, abandona)
  titulacion: Grado en Ingeniería Informática
  rama: TE
  sexo: Hombre
  via_acceso: Ciclo Formativo de Grado sup. o equivalente
  nota_acceso: 7.4
  nota_1er_anio: 6.42
  edad_entrada: 3.135
  n_anios_trabajando: 2
  abandono: 1

🟢 NO ABANDONA | C2 — Medicina (nota 11.94, no abandona)
  titulacion: Grado en Medicina
  rama: SA
  sexo: Mujer
  via_acceso: Pruebas acceso Bachiller Logse
  nota_acceso: 11.94
  nota_1er_anio: 7.54
  edad_entrada: 2.944
  n_anios_trabajando: 3
  abandono: 0

🔴 ABANDONA | C3 — Comunicación (nota alta, abandona)
  titulacion: Grado en Comunicación Audiovisual
  rama: SO
  sexo: Hombre
  via_acceso: Pruebas acceso Bachiller Logse
  nota_acceso: 8.57
  nota_1er_anio: 6.45
  edad_entrada: 2.996
  n_anios_trabajando: 4
  abandono: 1

🟢 NO ABANDONA | C4 — Ing. Informática (mujer, no abandona)
  titulacion: Grado en Ingeniería Informática
  rama: TE
  sexo: Mujer
  via_acceso: Pruebas acceso Bachiller Logse
  not

In [5]:
# ============================================================================
# CELDA 5: GUARDAR meta_test_app.parquet
# ============================================================================
RUTA_SALIDA = ROOT / "data" / "06_evaluacion" / "meta_test_app.parquet"
RUTA_SALIDA.parent.mkdir(parents=True, exist_ok=True)

df_app.to_parquet(RUTA_SALIDA, index=True)

print(f"✅ Guardado: {RUTA_SALIDA}")
print(f"   Shape:   {df_app.shape}")
print(f"   Tamaño:  {RUTA_SALIDA.stat().st_size / 1024:.1f} KB")

✅ Guardado: c:\PRUEBAS\AU_UJI_v2_RUTA_B\data\06_evaluacion\meta_test_app.parquet
   Shape:   (6725, 35)
   Tamaño:  216.0 KB
